# Tutorial3: GAT implementation

## Outline

- Implementation of GAT

Official resources:
* [Code](https://dsgiitr.com/blogs/gat/)

In [16]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

2.6.0+cu124
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [17]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Structure

In [18]:
class GATLayer(nn.Module):
    """
    Simple PyTorch Implementation of the Graph Attention layer.
    """
    def __init__(self):
        super(GATLayer, self).__init__()

    def forward(self, input, adj):
        print("")

## Let's start from the forward method

### Linear Transformation

$$
\bar{h'}_i = \textbf{W}\cdot \bar{h}_i
$$
with $\textbf{W}\in\mathbb R^{F'\times F}$ and $\bar{h}_i\in\mathbb R^{F}$.

$$
\bar{h'}_i \in \mathbb{R}^{F'}
$$

In [19]:
in_features = 5
out_features = 2
nb_nodes = 3

W = nn.Parameter(torch.zeros(size=(in_features, out_features))) #xavier paramiter inizializator
nn.init.xavier_uniform_(W.data, gain=1.414)

input = torch.rand(nb_nodes,in_features)


# linear transformation
h = torch.mm(input, W)
N = h.size()[0]

print(h.shape)

torch.Size([3, 2])


### Attention Mechanism

![title](https://github.com/AntonioLonga/PytorchGeometricTutorial/blob/main/Tutorial3/AttentionMechanism.png?raw=1)

In [20]:
a = nn.Parameter(torch.zeros(size=(2*out_features, 1))) #xavier paramiter inizializator
nn.init.xavier_uniform_(a.data, gain=1.414)
print(a.shape)

leakyrelu = nn.LeakyReLU(0.2)  # LeakyReLU

torch.Size([4, 1])


In [35]:
a_input = torch.cat([h.repeat(1, N).view(N * N, -1), h.repeat(N, 1)], dim=1).view(N, -1, 2 * out_features)


In [38]:
print(a_input)
print(a_input.shape)

tensor([[[-0.3730,  0.9874, -0.3730,  0.9874],
         [-0.3730,  0.9874, -0.3711,  0.8927],
         [-0.3730,  0.9874, -0.2397,  0.3906]],

        [[-0.3711,  0.8927, -0.3730,  0.9874],
         [-0.3711,  0.8927, -0.3711,  0.8927],
         [-0.3711,  0.8927, -0.2397,  0.3906]],

        [[-0.2397,  0.3906, -0.3730,  0.9874],
         [-0.2397,  0.3906, -0.3711,  0.8927],
         [-0.2397,  0.3906, -0.2397,  0.3906]]], grad_fn=<ViewBackward0>)
torch.Size([3, 3, 4])


![title](https://github.com/AntonioLonga/PytorchGeometricTutorial/blob/main/Tutorial3/a_input.png?raw=1)

In [22]:
e = leakyrelu(torch.matmul(a_input, a).squeeze(2))

In [40]:
print(torch.matmul(a_input,a))
print("")
print(torch.matmul(a_input,a).shape)
print("")
print(torch.matmul(a_input,a).squeeze(2).shape)
print("")
print(torch.matmul(a_input,a).squeeze(2))

tensor([[[ 0.8533],
         [ 0.7068],
         [-0.1709]],

        [[ 0.9458],
         [ 0.7993],
         [-0.0785]],

        [[ 1.4376],
         [ 1.2911],
         [ 0.4133]]], grad_fn=<UnsafeViewBackward0>)

torch.Size([3, 3, 1])

torch.Size([3, 3])

tensor([[ 0.8533,  0.7068, -0.1709],
        [ 0.9458,  0.7993, -0.0785],
        [ 1.4376,  1.2911,  0.4133]], grad_fn=<SqueezeBackward1>)


### Masked Attention

In [24]:
# Masked Attention
adj = torch.randint(2, (3, 3))

zero_vec  = -9e15*torch.ones_like(e)
print(zero_vec.shape)

torch.Size([3, 3])


In [42]:
attention = torch.where(adj > 0, e, zero_vec)
print(adj,"\n",e,"\n",zero_vec)
print(attention)

tensor([[0, 1, 1],
        [1, 0, 1],
        [0, 1, 0]]) 
 tensor([[ 0.8533,  0.7068, -0.0342],
        [ 0.9458,  0.7993, -0.0157],
        [ 1.4376,  1.2911,  0.4133]], grad_fn=<LeakyReluBackward0>) 
 tensor([[-9.0000e+15, -9.0000e+15, -9.0000e+15],
        [-9.0000e+15, -9.0000e+15, -9.0000e+15],
        [-9.0000e+15, -9.0000e+15, -9.0000e+15]])
tensor([[-9.0000e+15,  7.0685e-01, -3.4189e-02],
        [ 9.4579e-01, -9.0000e+15, -1.5700e-02],
        [-9.0000e+15,  1.2911e+00, -9.0000e+15]], grad_fn=<WhereBackward0>)


In [26]:
attention = F.softmax(attention, dim=1)
h_prime   = torch.matmul(attention, h)

In [27]:
attention

tensor([[0.0000, 0.6772, 0.3228],
        [0.7234, 0.0000, 0.2766],
        [0.0000, 1.0000, 0.0000]], grad_fn=<SoftmaxBackward0>)

In [28]:
h_prime

tensor([[-0.3286,  0.7306],
        [-0.3361,  0.8224],
        [-0.3711,  0.8927]], grad_fn=<MmBackward0>)

#### h_prime vs h

In [29]:
print(h_prime,"\n",h)

tensor([[-0.3286,  0.7306],
        [-0.3361,  0.8224],
        [-0.3711,  0.8927]], grad_fn=<MmBackward0>) 
 tensor([[-0.3730,  0.9874],
        [-0.3711,  0.8927],
        [-0.2397,  0.3906]], grad_fn=<MmBackward0>)


# Build the layer

In [30]:
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GATLayer, self).__init__()

        '''
        TODO
        '''

    def forward(self, input, adj):
        # Linear Transformation
        h = torch.mm(input, self.W) # matrix multiplication
        N = h.size()[0]

        # Attention Mechanism
        a_input = torch.cat([h.repeat(1, N).view(N * N, -1), h.repeat(N, 1)], dim=1).view(N, -1, 2 * self.out_features)
        e       = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))

        # Masked Attention
        zero_vec  = -9e15*torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)

        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)
        h_prime   = torch.matmul(attention, h)

        if self.concat:
            return F.elu(h_prime)
        else:
            return h_prime

In [31]:
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GATLayer, self).__init__()
        self.dropout       = dropout        # drop prob = 0.6
        self.in_features   = in_features    #
        self.out_features  = out_features   #
        self.alpha         = alpha          # LeakyReLU with negative input slope, alpha = 0.2
        self.concat        = concat         # conacat = True for all layers except the output layer.


        # Xavier Initialization of Weights
        # Alternatively use weights_init to apply weights of choice
        self.W = nn.Parameter(torch.zeros(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)

        self.a = nn.Parameter(torch.zeros(size=(2*out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

        # LeakyReLU
        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, input, adj):
        # Linear Transformation
        h = torch.mm(input, self.W) # matrix multiplication
        N = h.size()[0]
        print(N)

        # Attention Mechanism
        a_input = torch.cat([h.repeat(1, N).view(N * N, -1), h.repeat(N, 1)], dim=1).view(N, -1, 2 * self.out_features)
        e       = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))

        # Masked Attention
        zero_vec  = -9e15*torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)

        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)
        h_prime   = torch.matmul(attention, h)

        if self.concat:
            return F.elu(h_prime)
        else:
            return h_prime

# Use it

In [32]:
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

import matplotlib.pyplot as plt

name_data = 'Cora'
dataset = Planetoid(root= '/tmp/' + name_data, name = name_data)
dataset.transform = T.NormalizeFeatures()

print(f"Number of Classes in {name_data}:", dataset.num_classes)
print(f"Number of Node Features in {name_data}:", dataset.num_node_features)

Number of Classes in Cora: 7
Number of Node Features in Cora: 1433


Processing...
Done!


In [33]:
class GAT(torch.nn.Module):
    def __init__(self):
        super(GAT, self).__init__()
        self.hid = 8
        self.in_head = 8
        self.out_head = 1


        self.conv1 = GATConv(dataset.num_features, self.hid, heads=self.in_head, dropout=0.6)
        self.conv2 = GATConv(self.hid*self.in_head, dataset.num_classes, concat=False,
                             heads=self.out_head, dropout=0.6)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = "cpu"

model = GAT().to(device)
data = dataset[0].to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

model.train()
for epoch in range(1000):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

    if epoch%200 == 0:
        print(loss)

    loss.backward()
    optimizer.step()



tensor(1.9499, grad_fn=<NllLossBackward0>)
tensor(0.7430, grad_fn=<NllLossBackward0>)
tensor(0.6328, grad_fn=<NllLossBackward0>)
tensor(0.5720, grad_fn=<NllLossBackward0>)
tensor(0.5147, grad_fn=<NllLossBackward0>)


In [34]:
model.eval()
_, pred = model(data).max(dim=1)
correct = float(pred[data.test_mask].eq(data.y[data.test_mask]).sum().item())
acc = correct / data.test_mask.sum().item()
print('Accuracy: {:.4f}'.format(acc))

Accuracy: 0.8230
